<a href="https://colab.research.google.com/github/Anastasia-Marshakova/---/blob/main/%D0%9A%D0%BB%D0%BE%D0%BD_%D0%BB%D0%B0%D0%B1%D0%B0_1_mlops_colab_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import subprocess
from pathlib import Path

repo = Path("/content/mlops-labs-clean")

subprocess.run(
    [
        "git", "clone",
        "https://github.com/Anastasia-Marshakova/mlops-labs.git",
        str(repo),
    ],
    check=True,
)

CompletedProcess(args=['git', 'clone', 'https://github.com/Anastasia-Marshakova/mlops-labs.git', '/content/mlops-labs-clean'], returncode=0)

In [2]:
import os
import subprocess
import sys

os.chdir("/content/mlops-labs-clean")

for command in [
    [sys.executable, "-m", "pip", "install", "-q", "uv"],
    [
        "uv", "export", "--frozen", "--no-hashes",
        "--group", "dev",
        "-o", "/content/requirements-clean.lock.txt",
    ],
    [
        sys.executable, "-m", "pip", "install", "-q",
        "-r", "/content/requirements-clean.lock.txt",
    ],
]:
    subprocess.run(command, check=True)

In [4]:
from google.colab import userdata
import subprocess

token = userdata.get("DAGSHUB_TOKEN").strip()

for key in ("access_key_id", "secret_access_key"):
    result = subprocess.run(
        ["dvc", "remote", "modify", "--local", "origin", key, token],
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
    )
    if result.returncode != 0:
        print(result.stdout.replace(token, "[TOKEN HIDDEN]"))
        raise RuntimeError("Не удалось настроить DVC")

print("Доступ к хранилищу настроен")

Доступ к хранилищу настроен


In [5]:
import json
import subprocess
from pathlib import Path

expected = json.loads(
    Path("reports/metrics.json").read_text(encoding="utf-8")
)

for command in (["dvc", "pull"], ["dvc", "repro"]):
    result = subprocess.run(
        command,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
    )
    print(result.stdout)
    if result.returncode != 0:
        raise RuntimeError("Проверка остановлена")

actual = json.loads(
    Path("reports/metrics.json").read_text(encoding="utf-8")
)

assert actual == expected, "Метрики не совпали"

print("Метрики:", actual)
print("Проверка чистого клона пройдена")

A       data/features/test.csv
A       data/features/train.csv
A       data/processed/test.csv
A       data/processed/train.csv
A       data/raw/covertype.csv
A       models/model.joblib
6 files fetched and 6 files added

'data/raw/covertype.csv.dvc' didn't change, skipping
Stage 'prepare' didn't change, skipping
Stage 'featurize' didn't change, skipping
Stage 'train' didn't change, skipping
Stage 'evaluate' didn't change, skipping
Data and pipelines are up to date.

Метрики: {'accuracy': 0.8665, 'f1_macro': 0.7929977031433043, 'n_test': 4000}
Проверка чистого клона пройдена
